# Model Optimization: Pruning

In this notebook, we'll explore pruning techniques to reduce model size and improve inference speed. Pruning works by removing unnecessary weights from the model, effectively making it more sparse.

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM

# Import utility functions
from utils import measure_inference_time, get_model_size, measure_memory_usage, plot_comparison

## 2. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

# Load model information
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

## 3. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 4. Create Functions for Loading Models and Preparing Inputs

In [ ]:
def load_model(model_key):
    """Load model and tokenizer from local path."""
    model_data = model_info[model_key]
    model_path = model_data["local_path"]
    task = model_data["task"]
    
    print(f"Loading {model_data['model_name']} for {task}...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Load model based on task
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_path)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_path)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(model_path)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()  # Set model to evaluation mode
    
    return model, tokenizer, device

def prepare_inputs(model_key, tokenizer, device):
    """Prepare inputs for the model based on task."""
    task = model_info[model_key]["task"]
    sample_input = sample_inputs[model_key]
    
    if task == "sequence-classification" or task == "token-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "question-answering":
        inputs = tokenizer(sample_input["question"], sample_input["context"], return_tensors="pt")
    elif task == "masked-lm":
        inputs = tokenizer(sample_input, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    return inputs

## 5. Implement Magnitude-Based Pruning

Magnitude-based pruning removes weights with the smallest absolute values, as they contribute the least to the model's output.

In [ ]:
def prune_model_magnitude(model_key, pruning_ratio=0.3):
    """Apply magnitude-based pruning to a model."""
    # Load model and tokenizer
    model, tokenizer, device = load_model(model_key)
    
    # Prepare inputs
    inputs = prepare_inputs(model_key, tokenizer, device)
    
    # Create directory for pruned model
    model_name = model_info[model_key]["model_name"].replace("/", "_")
    pruned_dir = os.path.join("models", f"{model_name}_pruned_{int(pruning_ratio*100)}pct")
    os.makedirs(pruned_dir, exist_ok=True)
    
    print(f"Pruning {model_name} with {pruning_ratio*100}% pruning ratio...")
    
    # Apply magnitude-based pruning
    for name, param in model.named_parameters():
        if 'weight' in name and param.dim() > 1:  # Only prune weight matrices
            # Get the absolute values of the weights
            abs_weights = torch.abs(param.data)
            
            # Calculate the threshold for pruning
            threshold = torch.quantile(abs_weights.view(-1), pruning_ratio)
            
            # Create a mask for pruning
            mask = abs_weights > threshold
            
            # Apply the mask (set weights below threshold to zero)
            param.data = param.data * mask
    
    # Save pruned model
    model.save_pretrained(pruned_dir)
    tokenizer.save_pretrained(pruned_dir)
    
    # Measure metrics
    model_size = get_model_size(model)
    inference_time = measure_inference_time(model, inputs)
    memory_usage = measure_memory_usage(model, inputs)
    
    # Calculate sparsity
    total_params = 0
    zero_params = 0
    for name, param in model.named_parameters():
        if 'weight' in name and param.dim() > 1:
            total_params += param.numel()
            zero_params += (param.data == 0).sum().item()
    
    sparsity = zero_params / total_params if total_params > 0 else 0
    
    # Run inference to get output
    with torch.no_grad():
        outputs = model(**inputs)
    
    return {
        "model_key": model_key,
        "model_name": model_info[model_key]["model_name"],
        "task": model_info[model_key]["task"],
        "model_size": model_size,  # MB
        "inference_time": inference_time,  # ms
        "memory_usage": memory_usage,  # MB
        "sparsity": sparsity * 100,  # percentage
        "pruned_outputs": outputs,
        "pruned_path": pruned_dir
    }

## 6. Apply Pruning to Models

In [ ]:
# Apply pruning to each model
pruned_metrics = {}
pruning_ratio = 0.3  # 30% pruning

for model_key in model_info.keys():
    print(f"\nApplying pruning to {model_key}...")
    try:
        pruned_metrics[model_key] = prune_model_magnitude(model_key, pruning_ratio)
        
        # Print metrics
        print(f"Model size: {pruned_metrics[model_key]['model_size']:.2f} MB")
        print(f"Inference time: {pruned_metrics[model_key]['inference_time']:.2f} ms")
        print(f"Memory usage: {pruned_metrics[model_key]['memory_usage']:.2f} MB")
        print(f"Sparsity: {pruned_metrics[model_key]['sparsity']:.2f}%")
    except Exception as e:
        print(f"Error pruning {model_key}: {e}")

## 7. Compare Baseline vs. Pruned Models

In [ ]:
# Create comparison data
comparison_data = []

for model_key in pruned_metrics.keys():
    baseline = baseline_metrics[model_key]
    pruned = pruned_metrics[model_key]
    
    model_name = baseline["model_name"]
    
    comparison_data.append({
        "Model": model_name,
        "Metric": "Size (MB)",
        "Baseline": baseline["model_size"],
        "Pruned": pruned["model_size"],
        "Reduction (%)": (1 - pruned["model_size"] / baseline["model_size"]) * 100
    })
    
    comparison_data.append({
        "Model": model_name,
        "Metric": "Inference Time (ms)",
        "Baseline": baseline["inference_time"],
        "Pruned": pruned["inference_time"],
        "Reduction (%)": (1 - pruned["inference_time"] / baseline["inference_time"]) * 100
    })
    
    comparison_data.append({
        "Model": model_name,
        "Metric": "Memory Usage (MB)",
        "Baseline": baseline["memory_usage"],
        "Pruned": pruned["memory_usage"],
        "Reduction (%)": (1 - pruned["memory_usage"] / baseline["memory_usage"]) * 100
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df

In [ ]:
# Plot size comparison
size_df = comparison_df[comparison_df["Metric"] == "Size (MB)"]
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="value", hue="variable", data=pd.melt(size_df, id_vars=["Model", "Metric"], value_vars=["Baseline", "Pruned"]))
plt.title("Model Size Comparison: Baseline vs. Pruned")
plt.ylabel("Size (MB)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot inference time comparison
time_df = comparison_df[comparison_df["Metric"] == "Inference Time (ms)"]
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="value", hue="variable", data=pd.melt(time_df, id_vars=["Model", "Metric"], value_vars=["Baseline", "Pruned"]))
plt.title("Inference Time Comparison: Baseline vs. Pruned")
plt.ylabel("Inference Time (ms)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot memory usage comparison
memory_df = comparison_df[comparison_df["Metric"] == "Memory Usage (MB)"]
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="value", hue="variable", data=pd.melt(memory_df, id_vars=["Model", "Metric"], value_vars=["Baseline", "Pruned"]))
plt.title("Memory Usage Comparison: Baseline vs. Pruned")
plt.ylabel("Memory Usage (MB)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot reduction percentages
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Reduction (%)", hue="Metric", data=comparison_df)
plt.title("Reduction Percentages from Pruning")
plt.ylabel("Reduction (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Visualize Weight Distributions Before and After Pruning

In [ ]:
def plot_weight_distribution(model_key):
    """Plot weight distribution before and after pruning."""
    # Load original model
    original_model, _, _ = load_model(model_key)
    
    # Load pruned model
    pruned_path = pruned_metrics[model_key]["pruned_path"]
    task = model_info[model_key]["task"]
    
    if task == "sequence-classification":
        pruned_model = AutoModelForSequenceClassification.from_pretrained(pruned_path)
    elif task == "token-classification":
        pruned_model = AutoModelForTokenClassification.from_pretrained(pruned_path)
    elif task == "question-answering":
        pruned_model = AutoModelForQuestionAnswering.from_pretrained(pruned_path)
    elif task == "masked-lm":
        pruned_model = AutoModelForMaskedLM.from_pretrained(pruned_path)
    
    # Get weights from a specific layer (e.g., first linear layer)
    original_weights = None
    pruned_weights = None
    
    for name, param in original_model.named_parameters():
        if 'weight' in name and param.dim() > 1:
            original_weights = param.detach().cpu().numpy().flatten()
            break
    
    for name, param in pruned_model.named_parameters():
        if 'weight' in name and param.dim() > 1:
            pruned_weights = param.detach().cpu().numpy().flatten()
            break
    
    if original_weights is not None and pruned_weights is not None:
        plt.figure(figsize=(12, 6))
        
        # Plot histograms
        plt.subplot(1, 2, 1)
        plt.hist(original_weights, bins=100, alpha=0.7)
        plt.title("Original Weight Distribution")
        plt.xlabel("Weight Value")
        plt.ylabel("Frequency")
        
        plt.subplot(1, 2, 2)
        plt.hist(pruned_weights, bins=100, alpha=0.7)
        plt.title("Pruned Weight Distribution")
        plt.xlabel("Weight Value")
        plt.ylabel("Frequency")
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Plot weight distribution for a model
model_key = list(pruned_metrics.keys())[0]  # Choose the first model
plot_weight_distribution(model_key)

## 9. Save Pruned Metrics

In [ ]:
# Save pruned metrics (excluding outputs which aren't JSON serializable)
serializable_metrics = {}
for model_key, metrics in pruned_metrics.items():
    serializable_metrics[model_key] = {
        "model_key": metrics["model_key"],
        "model_name": metrics["model_name"],
        "task": metrics["task"],
        "model_size": metrics["model_size"],
        "inference_time": metrics["inference_time"],
        "memory_usage": metrics["memory_usage"],
        "sparsity": metrics["sparsity"],
        "pruned_path": metrics["pruned_path"]
    }

with open('pruned_metrics.json', 'w') as f:
    json.dump(serializable_metrics, f, indent=2)

print("Pruned metrics saved to pruned_metrics.json")

## 10. Estimate Cost Savings

In [ ]:
def estimate_monthly_cost(model_metrics, requests_per_month=1000000):
    """Estimate monthly cost for running a model in production."""
    # Assumptions
    compute_cost_per_hour = 0.5  # $0.5 per hour for compute (e.g., ml.g4dn.xlarge)
    storage_cost_per_gb_month = 0.023  # $0.023 per GB-month for S3
    
    # Calculate compute cost
    inference_time_hours = (model_metrics["inference_time"] * requests_per_month) / (1000 * 60 * 60)
    compute_cost = inference_time_hours * compute_cost_per_hour
    
    # Calculate storage cost
    storage_cost = (model_metrics["model_size"] / 1024) * storage_cost_per_gb_month
    
    # Total cost
    total_cost = compute_cost + storage_cost
    
    return {
        "compute_cost": compute_cost,
        "storage_cost": storage_cost,
        "total_cost": total_cost
    }

In [ ]:
# Estimate costs for baseline and pruned models
cost_savings = []

for model_key in pruned_metrics.keys():
    baseline_cost = estimate_monthly_cost(baseline_metrics[model_key])
    pruned_cost = estimate_monthly_cost(serializable_metrics[model_key])
    
    savings = {
        "Model": baseline_metrics[model_key]["model_name"],
        "Baseline Cost ($)": baseline_cost["total_cost"],
        "Pruned Cost ($)": pruned_cost["total_cost"],
        "Savings ($)": baseline_cost["total_cost"] - pruned_cost["total_cost"],
        "Savings (%)": (1 - pruned_cost["total_cost"] / baseline_cost["total_cost"]) * 100
    }
    
    cost_savings.append(savings)

savings_df = pd.DataFrame(cost_savings)
savings_df

In [ ]:
# Plot cost savings
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="value", hue="variable", data=pd.melt(savings_df, id_vars=["Model"], value_vars=["Baseline Cost ($)", "Pruned Cost ($)"]))
plt.title("Cost Comparison: Baseline vs. Pruned (1M requests/month)")
plt.ylabel("Monthly Cost ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot savings percentage
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Savings (%)", data=savings_df)
plt.title("Cost Savings Percentage from Pruning")
plt.ylabel("Savings (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Next Steps

In this notebook, we've applied pruning to our models and measured the impact on size, inference time, and memory usage. We've also estimated the cost savings from pruning.

In the next notebook, we'll explore knowledge distillation to create smaller, faster models that retain most of the accuracy of the original models.